<center><h1>Companhia Aberta Demonstrativo Financeiro</h1></center>

# <h2>Load Libraries</h2>

In [38]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 10)

# <h2>Constants</h2>

In [39]:
CD_CONTA = 'CD_CONTA'
DS_CONTA = 'DS_CONTA'

In [40]:
file_name = 'dfp_cia_aberta_BPA_con_2020.csv'
output_name = 'output.csv'

In [41]:
CNPJ_CIA = '97.837.181/0001-47'
ORDEM_EXERC = 'ÚLTIMO'

# <h2>Import Data</h2>

## <h3>Load .csv</h3>

In [42]:
try:
    cia_aberta_df = pd.read_csv(file_name, encoding='ISO-8859-1', sep=";")
except Exception as e:
    print(f"Error: {e}")

## <h3>Select Company</h3>

In [43]:
df = cia_aberta_df[cia_aberta_df['CNPJ_CIA'] == CNPJ_CIA].copy()    # seleciono somente as linhas relativas a uma companhia de interesse
df = df[df['ORDEM_EXERC'] == ORDEM_EXERC]                           # seleciono somente o último ou penúltimo exercício
df = df[[CD_CONTA, DS_CONTA,'VL_CONTA']]
df.reset_index(inplace=True, drop=True)

In [44]:
df

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0
1,1.01,Ativo Circulante,4220022.0
2,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Aplicações Financeiras,0.0
4,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...
71,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


# <h2>Wrangling</h2>

## <h3>Number of Steps</h3>

`CD_CONTA` is a column which contains codes. Each code was generated by chaining numbers and dots, like: `1`, `1.01`, `1.01.01`, `1.01.02`...

Each new dot in front of a number represents a sub hierarchy. For example: `1.01` is a subset of `1`, `1.01.01` is a subset of `1.01`, etc.

We want to find which is the maximum code lenght, which corrisponds to the total number of levels (ranks) in the hierarchy.

First, we need to split each code as a list of numbers: `1.01` thus becomes `[1, 01]`, `1.01.01` becomes `[1, 01, 01]`, etc.

We store this column of lists in a variable called `cd_conta_split`.
Then, we calculate the lengths of each one of these lists and store the result in a variable called `cd_conta_len`.
Finally, we find the number of ranks using `max`()

In [45]:
cd_conta_split = df[CD_CONTA].str.split('.')
cd_conta_len = [len(lst) for lst in cd_conta_split]    # len(row) or len(lst)?
num_levels = max(cd_conta_len)

In [46]:
num_levels

5

## <h3>For Loop</h3>

In [48]:
rounds = range(1, num_levels+1)    # rounds_range? range_rounds?

for round in rounds:

    """
    ### JOIN

    We want to find which codes have minimum lenght. The minimum length should correspond to the round number
    (the minimum length at round `1` should be `1`, at round `2` should be `2`, etc.).

    First, we need to split each code as a list of numbers: `1.01` thus becomes `[1, 01]`, `1.01.01` becomes `[1, 01, 01]`, etc.

    We store this column of lists in a variable called `cd_conta_split`.
    Then, we calculate the lengths of each one of these lists and store the result in a variable called `cd_conta_len`.
    """

    cd_conta_split = df[CD_CONTA].str.split('.')

    cd_conta_len = [len(row) for row in cd_conta_split]        # len(row) or len(lst)?

    """
    Now that we have a column of code lengths named `cd_conta_len`, we just need to find which index corresponds to the codes with minimum length (i.e., with lenght == round)
    """

    idx_list = [idx for idx, element in enumerate(cd_conta_len) if element == round]

    """
    Now that we have a list of the indexes of the codes which correspond to the highest rank, we need to join the corrisponding description to all the indexes of lowest rank.
    For example: the description corrisponding to `1` will be assigned to all the codes which start with `1` (`1.01` , `1.01.01`...), the description corrisponding to `2` will be assigned to all the codes which start with `2`, et cetera.

    We'll first store a list of the join keys in the variable `keys` and then create a dataframe named `key_desc_dict` with the join keys as keys and the description extracted from `DS_CONTA` as values.
    """

    keys = df[CD_CONTA][idx_list]
    key_desc_dict = {key: df[DS_CONTA][idx] for key, idx in zip(keys, idx_list)}

    """
    The join operation is performed using a list comprehension. For any row in the column `cd_conta_split`, a key is obtained.

    The keys are extracted as the first `i` elements of each code, where `i` is the round number. In practice, during the first round, for example, all codes starting with `1` will produce keys equal to `1` (length == 1).
    During the second round, all codes starting with `1.01` will give keys equal to `1.01` (length==2), etc.

    For any row in `cd_conta_split`, the key is compared to the join keys stored in `key_desc_dict`. If they are equal, the value (description) corresponding to the key is returned.

    In practice, if the join key is `1` and the key of a certain row of the join column is `1`, the description stored in `DS_CONTA` corresponding to the value `1` in `CD_CONTA` will be assigned to that row.
    """

    ds_conta = [key_desc_dict[key]
                for row in cd_conta_split
                for key in key_desc_dict
                if ".".join(row[:round]) == key]

    """
    ### Insert:

    Check if the column you're about to insert is equal to `DS_CONTA`.
    If it's equal, you've already finished rearranging the dataframe.
    """

    colname=DS_CONTA + '_' + str(round)

    if pd.Series(ds_conta).equals(df[DS_CONTA]):
        df.rename(columns={DS_CONTA: colname}, inplace=True)
        df.to_csv(output_name, index=False)
        exit
    else:
        df.insert(round, column=colname, value=ds_conta)

    """
    ### Drop rows:

    We want to drop the rows corresponding to the `CD_CONTA` which was used to join (the `CD_CONTA` with minimum length), but only if there are more sub-hierarchies, more subcategories of this code.

    Firstly, we select all the rows which code was used to join (variable `idx_list`).
    For each key which was used to join, we check if there is at least another row in CD_CONTA which code starts with the same key (example: CD_CONTA 1.01 starting with 1; CD_CONTA 1.01.01 starting with 1.01, etc.)

    If there is at least one, we can safely delete the row.

    Otherwise, we keep the row and add .00 to the code in CD_CONTA, so the code can have the correct length in the next round (length 2 in round 2, length 3 in round 3, etc.)

    """

    for idx in idx_list:
        mask = [True]*len(ds_conta)
        mask[idx] = False
        if any(".".join(row[:round]) == df.loc[idx, CD_CONTA] for row in cd_conta_split[mask]):
            df = df.drop(idx)
        else:
            df.loc[idx, CD_CONTA] = df.loc[idx, CD_CONTA] + '.00'

    df.reset_index(inplace=True, drop=True)

CPU times: user 89.4 ms, sys: 865 µs, total: 90.2 ms
Wall time: 90.6 ms


In [49]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA_5,VL_CONTA
0,1.01.01.00.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,1.01.02.01.01.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,1.01.02.01.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,1.01.02.02.00.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.03.00.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...
49,1.02.04.02.07.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,1.02.04.02.08.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,1.02.04.02.09.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,1.02.04.02.10.00,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0
